#  DermaVision AI  MobileNet | Full Metrics + GradCAM++ + All Saves
**Dataset:** HAM10000 Skin Cancer | **Model:** MobileNet (Transfer Learning)  
**Outputs:** Metrics CSV · Accuracy/Loss Curve · Confusion Matrix · ROC Curve · GradCAM++ Images

##  Cell 1  Imports

In [1]:
import matplotlib
matplotlib.use('Agg')  # Headless  saves to file instead of showing popup

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import pandas as pd
import numpy as np
import os, time, json
from glob import glob
from pathlib import Path

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_score,
    recall_score, f1_score, accuracy_score
)

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {tf.config.list_physical_devices("GPU")}')

# ─── Output directory ───────────────────────────────────────────────────────
OUTPUT_DIR = Path('kaggle/working/dermavision_outputs')
for sub in ['curves', 'confusion_matrix', 'roc_curve', 'gradcam', 'metrics', 'model']:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'\n Output directory ready: {OUTPUT_DIR}')

2026-05-16 10:07:49.372217: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-16 10:07:49.413784: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-16 10:07:49.413845: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-16 10:07:49.417079: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-16 10:07:49.428157: I tensorflow/core/platform/cpu_feature_guar

TensorFlow version: 2.15.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

 Output directory ready: kaggle/working/dermavision_outputs


2026-05-16 10:07:51.994419: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-16 10:07:52.058236: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-16 10:07:52.058305: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-16 10:07:52.058313: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


# Config MLFlow

In [2]:
import mlflow
import datetime


class MLflowEpochLogger(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            return
        # Registra cada métrica con el número de época como step
        mlflow.log_metric("train_loss",     logs.get("loss"),         step=epoch)
        mlflow.log_metric("train_accuracy", logs.get("accuracy"),     step=epoch)
        mlflow.log_metric("val_loss",       logs.get("val_loss"),     step=epoch)
        mlflow.log_metric("val_accuracy",   logs.get("val_accuracy"), step=epoch)

# MLFLOW Parametro clave
parametros_clave = {
    "batch_size": 16,
    "epochs": 15, # o las que tengas configuradas
    "trainable": False
}

##  Cell 2  Data Loading & EDA

In [3]:
base_skin_dir = 'kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000'

imageid_path_dict = {
    os.path.splitext(os.path.basename(x))[0]: x
    for x in glob(os.path.join(base_skin_dir, '*', '*.jpg'))
}

skin_df = pd.read_csv(os.path.join(base_skin_dir, 'HAM10000_metadata.csv'))
skin_df['path'] = skin_df['image_id'].map(imageid_path_dict.get)

LABEL_MAP = {
    'nv':    'Melanocytic nevi',
    'mel':   'Melanoma',
    'bkl':   'Benign keratosis-like lesions',
    'bcc':   'Basal cell carcinoma',
    'akiec': 'Actinic keratoses',
    'vasc':  'Vascular lesions',
    'df':    'Dermatofibroma'
}
skin_df['cell_type'] = skin_df['dx'].map(LABEL_MAP)
NUM_CLASSES = len(LABEL_MAP)

print(f'Total images found : {len(skin_df)}')
print(f'Classes            : {NUM_CLASSES}')
print(skin_df['cell_type'].value_counts())
skin_df.head()

Total images found : 10015
Classes            : 7
cell_type
Melanocytic nevi                 6705
Melanoma                         1113
Benign keratosis-like lesions    1099
Basal cell carcinoma              514
Actinic keratoses                 327
Vascular lesions                  142
Dermatofibroma                    115
Name: count, dtype: int64


,lesion_id,image_id,dx,dx_type,age,sex,localization,path,cell_type
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,kaggle/input/datasets/kmader/skin-cancer-mnist...,Benign keratosis-like lesions
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,kaggle/input/datasets/kmader/skin-cancer-mnist...,Benign keratosis-like lesions
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,kaggle/input/datasets/kmader/skin-cancer-mnist...,Benign keratosis-like lesions
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,kaggle/input/datasets/kmader/skin-cancer-mnist...,Benign keratosis-like lesions
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,kaggle/input/datasets/kmader/skin-cancer-mnist...,Benign keratosis-like lesions


##  Cell 3  Balancing + Train/Val/Test Split

In [4]:
# 1 Split primero (sobre datos originales, sin duplicados)
train_df, dummy_df = train_test_split(skin_df, test_size=0.30, random_state=42, stratify=skin_df['dx'])
val_df, test_df   = train_test_split(dummy_df, test_size=0.50, random_state=42, stratify=dummy_df['dx'])

# 2 Oversample SOLO el train
# Contar todas las clases ordenadas de mayor a menor
train_df['dx'].value_counts()

# Cada clase dentro de train_df se aumentara a la cantidad de elementos que ya tenga la clase mayoritaria
clase_mayoritaria = train_df['dx'].value_counts().idxmax()
cantidad          = train_df['dx'].value_counts().max()
print(f"Clase mayoritaria: {clase_mayoritaria} → {cantidad} muestras")
MAX_SAMPLES = cantidad
train_balanced = pd.concat([
    resample(train_df[train_df['dx'] == cat], replace=True, n_samples=MAX_SAMPLES, random_state=42)
    for cat in train_df['dx'].unique()
])
# val_df y test_df se quedan sin tocar 


# Imprimo resultados de los conjuntos
print(f'Train : {train_balanced.shape}')
print(f'Val   : {val_df.shape}')
print(f'Test  : {test_df.shape}')


# Compruebo si hay duplicados
train_ids = set(train_df["image_id"])
test_ids = set(test_df["image_id"])
duplicados = train_ids.intersection(test_ids)
print(f"Número de image_id repetidos: {len(duplicados)}")
print(duplicados)

Clase mayoritaria: nv → 4693 muestras
Train : (32851, 9)
Val   : (1502, 9)
Test  : (1503, 9)
Número de image_id repetidos: 0
set()


##  Cell 4  Data Generators

In [5]:
IMG_SIZE   = (224, 224)

# Training generator  with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

# Val / Test  only rescale
test_datagen = ImageDataGenerator(rescale=1./255)

def make_gen(datagen, df, shuffle=True):
    return datagen.flow_from_dataframe(
        df, x_col='path', y_col='cell_type',
        target_size=IMG_SIZE, batch_size=parametros_clave["batch_size"],
        class_mode='categorical', shuffle=shuffle
    )

train_gen = make_gen(train_datagen, train_df, shuffle=True)
val_gen   = make_gen(test_datagen,  val_df,   shuffle=False)
test_gen  = make_gen(test_datagen,  test_df,  shuffle=False)

CLASS_NAMES = list(train_gen.class_indices.keys())
print('Class order:', CLASS_NAMES)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.
Class order: ['Actinic keratoses', 'Basal cell carcinoma', 'Benign keratosis-like lesions', 'Dermatofibroma', 'Melanocytic nevi', 'Melanoma', 'Vascular lesions']


In [6]:
from keras_cv_attention_models import coatnet

# ─── Base Model ─────────────────────────────────────────────────────────────
base_model = coatnet.CoAtNet3(num_classes=0)
base_model.trainable = False   # Freeze for transfer learning

# ─── Custom Head ────────────────────────────────────────────────────────────
x = base_model.output
x = GlobalAveragePooling2D()(x)   # GAP  needed for GradCAM++ compatibility
x = BatchNormalization()(x)
x = Dense(1024, activation='relu', name='fc1')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu', name='fc2')(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# GradCAM++ target layer = last conv layer of MobileNet
GRADCAM_LAYER = 'conv_pw_13_relu'

print(f'Total params  : {model.count_params():,}')
print(f'GradCAM layer : {GRADCAM_LAYER}')

2026-05-16 10:07:52.838967: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-16 10:07:52.839195: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-16 10:07:52.839288: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-16 10:07:52.839300: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
2026-05-16 10:07:53.644927: I external/lo

Total params  : 160,492,515
GradCAM layer : conv_pw_13_relu


##  Cell 5  MobileNet Model Build

##  Cell 6  Training

In [7]:
# MLFLOW. Se indica al cuaderno a dónde enviar los datos. Si tu MLflow local está en el puerto por defecto, sería así:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("MobileNet_CoAtNet")

callbacks = [
    EarlyStopping(monitor='val_loss', patience=6,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2,
                      patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(
        str(OUTPUT_DIR / 'model' / 'best_coatnet.h5'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    MLflowEpochLogger()
]

# MLFLOW Capturamos el momento exacto (Día, Mes, Hora y Minuto) y arrancamos la ejecución
marca_tiempo = datetime.datetime.now().strftime("%d%m_%H%M")
mlflow.start_run(run_name = f"coatnet3_sin_data_leakage_{marca_tiempo}")
RUN_ID = mlflow.active_run().info.run_id

history = model.fit(
    train_gen,
    steps_per_epoch  = train_gen.samples  // parametros_clave["batch_size"],
    validation_data  = val_gen,
    validation_steps = val_gen.samples    // parametros_clave["batch_size"],
    epochs           = parametros_clave["epochs"],
    callbacks        = callbacks,
    verbose          = 1
)

# Save full model
model.save(str(OUTPUT_DIR / 'model' / f'dermavision_{marca_tiempo}.h5'))
print('\n Model saved.')

Epoch 1/15


2026-05-16 10:08:05.623203: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
2026-05-16 10:08:05.746889: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-05-16 10:08:05.754318: W external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:225] Falling back to the CUDA driver for PTX compilation; ptxas does not support CC 12.0
2026-05-16 10:08:05.754381: W external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:228] Used ptxas at /home/jesus/repos/TFM/.venv/lib/python3.11/site-packages/tensorflow/python/platform/../../../nvidia/cuda_nvcc/bin/ptxas
2026-05-16 10:08:05.754492: W external/local_xla/xla/stream_executor/gpu/redzone_allocator.cc:322] UNIMPLEMENTED: /home/jesus/repos/TFM/.venv/lib/python3.11/site-packages/tensorflow/python/platform/../../../nvidia/cuda_nvcc/bin/ptxas ptxas too old. Falling back to the driver to compile.
Relying on driver to perform ptx compila

 65/438 [===>..........................] - ETA: 3:37 - loss: 2.1886 - accuracy: 0.4971

2026-05-16 10:08:48.712469: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.67GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-05-16 10:08:48.899236: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.67GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


438/438 [==============================] - ETA: 0s - loss: 1.8070 - accuracy: 0.5508
Epoch 1: val_accuracy improved from -inf to 0.67070, saving model to kaggle/working/dermavision_outputs/model/best_coatnet.h5


/home/jesus/repos/TFM/.venv/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


438/438 [==============================] - 328s 716ms/step - loss: 1.8070 - accuracy: 0.5508 - val_loss: 1.1218 - val_accuracy: 0.6707 - lr: 0.0010
Epoch 2/15
438/438 [==============================] - ETA: 0s - loss: 1.3333 - accuracy: 0.6333
Epoch 2: val_accuracy did not improve from 0.67070
438/438 [==============================] - 305s 697ms/step - loss: 1.3333 - accuracy: 0.6333 - val_loss: 1.1813 - val_accuracy: 0.6707 - lr: 0.0010
Epoch 3/15
438/438 [==============================] - ETA: 0s - loss: 1.2172 - accuracy: 0.6630
Epoch 3: val_accuracy did not improve from 0.67070
438/438 [==============================] - 305s 696ms/step - loss: 1.2172 - accuracy: 0.6630 - val_loss: 1.2189 - val_accuracy: 0.6700 - lr: 0.0010
Epoch 4/15
438/438 [==============================] - ETA: 0s - loss: 1.1612 - accuracy: 0.6676
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.

Epoch 4: val_accuracy did not improve from 0.67070
438/438 [============================

##  Cell 7  Accuracy & Loss Curves (Train + Validation) → Saved

In [8]:
def plot_accuracy_loss(history, save_path):
    """Plot and save Accuracy + Loss curves (train & validation) side-by-side."""
    acc     = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss    = history.history['loss']
    val_loss= history.history['val_loss']
    epochs  = range(1, len(acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('DermaVision AI  MobileNet Training Curves', fontsize=16, fontweight='bold')

    # ── Accuracy ──
    axes[0].plot(epochs, acc,     'b-o', linewidth=2, markersize=4, label='Train Accuracy')
    axes[0].plot(epochs, val_acc, 'r-o', linewidth=2, markersize=4, label='Val Accuracy')
    axes[0].set_title('Accuracy Curve', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(fontsize=12); axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0, 1])
    # Annotate best val accuracy
    best_ep = np.argmax(val_acc)
    axes[0].annotate(f'Best: {val_acc[best_ep]:.4f}',
                     xy=(epochs[best_ep], val_acc[best_ep]),
                     xytext=(epochs[best_ep]+0.5, val_acc[best_ep]-0.05),
                     arrowprops=dict(arrowstyle='->', color='red'),
                     fontsize=10, color='red')

    # ── Loss ──
    axes[1].plot(epochs, loss,     'b-o', linewidth=2, markersize=4, label='Train Loss')
    axes[1].plot(epochs, val_loss, 'r-o', linewidth=2, markersize=4, label='Val Loss')
    axes[1].set_title('Loss Curve', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(fontsize=12); axes[1].grid(True, alpha=0.3)
    # Annotate best val loss
    best_loss_ep = np.argmin(val_loss)
    axes[1].annotate(f'Best: {val_loss[best_loss_ep]:.4f}',
                     xy=(epochs[best_loss_ep], val_loss[best_loss_ep]),
                     xytext=(epochs[best_loss_ep]+0.5, val_loss[best_loss_ep]+0.05),
                     arrowprops=dict(arrowstyle='->', color='red'),
                     fontsize=10, color='red')

    plt.tight_layout()
    mlflow.log_figure(fig, "graficas/curvas.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f' Accuracy + Loss curves saved → {save_path}')

plot_accuracy_loss(history, OUTPUT_DIR / 'curves' / 'accuracy_loss_curve.png')

 Accuracy + Loss curves saved → kaggle/working/dermavision_outputs/curves/accuracy_loss_curve.png


##  Cell 8  Full Metrics: Accuracy, Precision, Recall, F1, AUC, Inference/Image

In [9]:
# ─── Predict on test set ─────────────────────────────────────────────────────
test_gen.reset()

# Measure inference time
N_WARMUP = 5
dummy_batch = next(iter(test_gen))[0][:1]   # 1 image
for _ in range(N_WARMUP):                   # warm-up GPU
    model.predict(dummy_batch, verbose=0)

# Time over 100 single images
times = []
for _ in range(100):
    t0 = time.perf_counter()
    model.predict(dummy_batch, verbose=0)
    times.append(time.perf_counter() - t0)
inference_ms = np.mean(times) * 1000   # milliseconds per image

# Full test predictions
Y_prob  = model.predict(test_gen, verbose=1)   # (N, 7) softmax probabilities
y_pred  = np.argmax(Y_prob, axis=1)
y_true  = test_gen.classes

# ─── Metrics ─────────────────────────────────────────────────────────────────
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_true, y_pred, average='weighted', zero_division=0)

# One-vs-Rest AUC (macro average)
from sklearn.preprocessing import label_binarize
y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
auc_scores = []
for i in range(NUM_CLASSES):
    if y_true_bin[:, i].sum() > 0:
        fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], Y_prob[:, i])
        auc_scores.append(auc(fpr_i, tpr_i))
macro_auc = np.mean(auc_scores)

# ─── Print summary ───────────────────────────────────────────────────────────
metrics_dict = {
    'Accuracy'          : round(accuracy, 4),
    'Precision (W)'     : round(precision, 4),
    'Recall (W)'        : round(recall, 4),
    'F1 Score (W)'      : round(f1, 4),
    'AUC Score (Macro)' : round(macro_auc, 4),
    'Inference/Image (ms)': round(inference_ms, 3)
}

print('\n' + '='*50)
print('       DermaVision AI  Test Metrics')
print('='*50)
for k, v in metrics_dict.items():
    print(f'  {k:<25} : {v}')
print('='*50)

# Save metrics to JSON
metrics_json_path = OUTPUT_DIR / 'metrics' / 'test_metrics.json'
with open(metrics_json_path, 'w') as f:
    json.dump(metrics_dict, f, indent=4)
    mlflow.log_artifact(metrics_json_path, artifact_path="evaluacion")
print(f'\n Metrics saved → {metrics_json_path}')

# Save per-class classification report to CSV
report_dict = classification_report(y_true, y_pred,
                                    target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()
report_csv_path = OUTPUT_DIR / 'metrics' / 'classification_report.csv'
report_df.to_csv(report_csv_path)

mlflow.log_artifact(report_csv_path, artifact_path="evaluacion")
print(f' Classification report saved → {report_csv_path}')


# Save training history to CSV
hist_df = pd.DataFrame(history.history)
hist_df.index.name = 'epoch'
hist_csv_path = OUTPUT_DIR / 'metrics' / 'training_history.csv'
hist_df.to_csv(hist_csv_path)
print(f' Training history saved → {hist_csv_path}')

2026-05-16 10:44:09.926074: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.63GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-05-16 10:44:10.016106: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.63GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


93/94 [============================>.] - ETA: 0s

2026-05-16 10:45:17.386054: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 5.22GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-05-16 10:45:18.581007: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 5.22GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


94/94 [==============================] - 58s 585ms/step

       DermaVision AI  Test Metrics
  Accuracy                  : 0.6693
  Precision (W)             : 0.448
  Recall (W)                : 0.6693
  F1 Score (W)              : 0.5367
  AUC Score (Macro)         : 0.6753
  Inference/Image (ms)      : 99.671

 Metrics saved → kaggle/working/dermavision_outputs/metrics/test_metrics.json
 Classification report saved → kaggle/working/dermavision_outputs/metrics/classification_report.csv
 Training history saved → kaggle/working/dermavision_outputs/metrics/training_history.csv


## 🟦 Cell 9  Confusion Matrix → Saved

In [10]:
def plot_confusion_matrix(y_true, y_pred, class_names, save_path):
    """Plot and save a styled confusion matrix."""
    cm_arr = confusion_matrix(y_true, y_pred)
    # Normalize for percentage annotation
    cm_norm = cm_arr.astype('float') / cm_arr.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(22, 8))
    fig.suptitle('Confusion Matrix  DermaVision AI (MobileNet)',
                 fontsize=16, fontweight='bold')

    # Raw counts
    sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[0], linewidths=0.5)
    axes[0].set_title('Raw Counts', fontsize=13)
    axes[0].set_ylabel('True Label', fontsize=11)
    axes[0].set_xlabel('Predicted Label', fontsize=11)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].tick_params(axis='y', rotation=0)

    # Normalized (%)
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
    axes[1].set_title('Normalized (Row %)', fontsize=13)
    axes[1].set_ylabel('True Label', fontsize=11)
    axes[1].set_xlabel('Predicted Label', fontsize=11)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].tick_params(axis='y', rotation=0)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    mlflow.log_figure(fig, "graficas/confusion_matrix.png")
    mlflow.log_artifact(save_path, artifact_path="evaluacion")
    plt.close()
    print(f' Confusion matrix saved → {save_path}')

plot_confusion_matrix(
    y_true, y_pred, CLASS_NAMES,
    OUTPUT_DIR / 'confusion_matrix' / 'confusion_matrix.png'
)

 Confusion matrix saved → kaggle/working/dermavision_outputs/confusion_matrix/confusion_matrix.png


##  Cell 10  ROC Curve (per class + Macro) → Saved

In [11]:
def plot_roc_curves(y_true, Y_prob, class_names, n_classes, save_path):
    """Plot and save per-class + macro ROC curves."""
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    fpr_all, tpr_all, roc_auc = {}, {}, {}
    for i in range(n_classes):
        fpr_all[i], tpr_all[i], _ = roc_curve(y_true_bin[:, i], Y_prob[:, i])
        roc_auc[i] = auc(fpr_all[i], tpr_all[i])

    # Macro average
    all_fpr = np.unique(np.concatenate([fpr_all[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr_all[i], tpr_all[i])
    mean_tpr /= n_classes
    macro_auc_val = auc(all_fpr, mean_tpr)

    # ── Plot ──
    palette = plt.cm.get_cmap('tab10', n_classes)
    fig, ax = plt.subplots(figsize=(12, 9))

    for i, cls in enumerate(class_names):
        ax.plot(fpr_all[i], tpr_all[i], color=palette(i), lw=1.8,
                label=f'{cls}  (AUC = {roc_auc[i]:.3f})')

    ax.plot(all_fpr, mean_tpr, 'k--', lw=2.5,
            label=f'Macro Avg  (AUC = {macro_auc_val:.3f})')
    ax.plot([0,1],[0,1], 'gray', lw=1, linestyle=':', label='Random Classifier')

    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel('False Positive Rate', fontsize=13)
    ax.set_ylabel('True Positive Rate', fontsize=13)
    ax.set_title('ROC Curves  DermaVision AI (MobileNet)',
                 fontsize=15, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9.5)
    ax.grid(True, alpha=0.25)

    plt.tight_layout()
    mlflow.log_figure(fig, "graficas/roc_curve.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

    # Save per-class AUC to CSV
    auc_df = pd.DataFrame({
        'Class': class_names,
        'AUC'  : [round(roc_auc[i], 4) for i in range(n_classes)]
    })
    auc_df.loc[len(auc_df)] = ['Macro Average', round(macro_auc_val, 4)]
    auc_csv = save_path.parent / 'per_class_auc.csv'
    auc_df.to_csv(auc_csv, index=False)
    print(f' ROC curve saved → {save_path}')
    print(f' Per-class AUC saved → {auc_csv}')

plot_roc_curves(
    y_true, Y_prob, CLASS_NAMES, NUM_CLASSES,
    OUTPUT_DIR / 'roc_curve' / 'roc_curve.png'
)

/tmp/ipykernel_832906/218807891.py:19: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  palette = plt.cm.get_cmap('tab10', n_classes)


 ROC curve saved → kaggle/working/dermavision_outputs/roc_curve/roc_curve.png
 Per-class AUC saved → kaggle/working/dermavision_outputs/roc_curve/per_class_auc.csv


##  Cell 11  GradCAM++ (Explainable AI) → Saved

In [12]:
# ─── GradCAM++ Implementation ────────────────────────────────────────────────

def get_gradcam_pp(model, img_array, layer_name, class_idx=None):
    """
    Compute GradCAM++ heatmap.
    Returns: heatmap (H, W) in [0, 1]
    """
    grad_model = Model(
        inputs  = model.input,
        outputs = [model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape2:
        with tf.GradientTape() as tape1:
            with tf.GradientTape() as tape0:
                conv_out, preds = grad_model(img_array, training=False)
                if class_idx is None:
                    class_idx = tf.argmax(preds[0])
                score = preds[:, class_idx]
            grads1 = tape0.gradient(score, conv_out)   # 1st derivative
        grads2 = tape1.gradient(grads1, conv_out)       # 2nd derivative
    grads3 = tape2.gradient(grads2, conv_out)           # 3rd derivative

    # GradCAM++ alpha weights
    denom       = 2.0 * grads2 + tf.reduce_sum(grads3 * conv_out,
                                                axis=(1, 2), keepdims=True)
    denom       = tf.where(denom == 0, tf.ones_like(denom), denom)
    alphas      = grads2 / denom

    relu_grads  = tf.nn.relu(score) * grads1
    weights     = tf.reduce_sum(alphas * relu_grads, axis=(1, 2))

    conv_out_np = conv_out[0].numpy()
    weights_np  = weights[0].numpy()

    cam = np.zeros(conv_out_np.shape[:2], dtype=np.float32)
    for i, w in enumerate(weights_np):
        cam += w * conv_out_np[:, :, i]

    cam = np.maximum(cam, 0)
    cam = cam / (cam.max() + 1e-8)
    return cam, int(class_idx)


def overlay_gradcam(img_orig, heatmap, alpha=0.45):
    """Overlay GradCAM++ heatmap on the original image."""
    heatmap_resized = np.uint8(255 * heatmap)
    heatmap_colored = cm.jet(heatmap_resized)[:, :, :3]  # RGBA → RGB
    heatmap_colored = np.uint8(heatmap_colored * 255)

    # Resize heatmap to match original image
    from PIL import Image as PILImage
    heatmap_pil = PILImage.fromarray(heatmap_colored).resize(
        (img_orig.shape[1], img_orig.shape[0]), PILImage.LANCZOS
    )
    heatmap_resized = np.array(heatmap_pil)

    overlay = (1 - alpha) * img_orig + alpha * heatmap_resized
    return np.uint8(np.clip(overlay, 0, 255))


def run_gradcam_on_test(model, test_gen, class_names, layer_name,
                         save_dir, n_per_class=2):
    """
    Run GradCAM++ on n_per_class images for every class.
    Saves individual images + a summary grid.
    """
    save_dir = Path(save_dir)
    class_idx_map = {v: k for k, v in test_gen.class_indices.items()}
    n_classes = len(class_names)

    # Collect indices per class
    all_indices = np.arange(len(test_gen.filenames))
    class_sample_indices = {}
    for cls_idx in range(n_classes):
        cls_mask = test_gen.classes == cls_idx
        pool = all_indices[cls_mask]
        chosen = pool[:n_per_class] if len(pool) >= n_per_class else pool
        class_sample_indices[cls_idx] = chosen

    # ── Build summary grid ──
    n_cols  = n_per_class * 3   # original | heatmap | overlay
    n_rows  = n_classes
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 3, n_rows * 3.2))
    fig.suptitle('GradCAM++  DermaVision AI (MobileNet)\n'
                 'Original | Heatmap | Overlay  (per class)',
                 fontsize=14, fontweight='bold')

    for row_idx, (cls_idx, sample_idxs) in enumerate(class_sample_indices.items()):
        cls_name = class_idx_map[cls_idx]

        for col_group, sample_idx in enumerate(sample_idxs):
            # Load image
            img_path = test_gen.filenames[sample_idx]
            img_pil  = load_img(img_path, target_size=(224, 224))
            img_np   = img_to_array(img_pil)        # uint8 [0,255]
            img_inp  = img_np / 255.0               # float [0,1]
            img_inp_exp = np.expand_dims(img_inp, 0).astype('float32')

            # GradCAM++
            heatmap, pred_idx = get_gradcam_pp(model, img_inp_exp,
                                               layer_name, cls_idx)
            overlay = overlay_gradcam(img_np, heatmap)

            # Heatmap colored (for display)
            hm_colored = np.uint8(cm.jet(
                np.uint8(255 * heatmap))[:, :, :3] * 255)

            # Save individual images
            prefix = save_dir / f'cls{cls_idx}_{cls_name.replace(" ","_")}_img{col_group}'
            plt.imsave(str(prefix) + '_original.png', img_np.astype(np.uint8))
            plt.imsave(str(prefix) + '_heatmap.png',  hm_colored)
            plt.imsave(str(prefix) + '_overlay.png',  overlay)

            # Fill grid
            base_col = col_group * 3
            for ax in axes[row_idx][base_col:base_col+3]:
                ax.axis('off')

            axes[row_idx][base_col].imshow(img_np.astype(np.uint8))
            axes[row_idx][base_col].set_title('Original' if row_idx==0 else '', fontsize=8)

            axes[row_idx][base_col+1].imshow(hm_colored)
            axes[row_idx][base_col+1].set_title('GradCAM++' if row_idx==0 else '', fontsize=8)

            axes[row_idx][base_col+2].imshow(overlay)
            axes[row_idx][base_col+2].set_title('Overlay' if row_idx==0 else '', fontsize=8)

        # Row label
        axes[row_idx][0].set_ylabel(cls_name, fontsize=9,
                                    fontweight='bold', labelpad=5)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    grid_path = save_dir / 'gradcam_summary_grid.png'
    plt.savefig(str(grid_path), dpi=130, bbox_inches='tight')
    plt.close()
    print(f' GradCAM++ summary grid saved → {grid_path}')
    print(f' Individual GradCAM++ images saved in → {save_dir}')


# ─── Run ─────────────────────────────────────────────────────────────────────
test_gen.reset()
run_gradcam_on_test(
    model      = model,
    test_gen   = test_gen,
    class_names= CLASS_NAMES,
    layer_name = GRADCAM_LAYER,
    save_dir   = OUTPUT_DIR / 'gradcam',
    n_per_class= 2          # 2 sample images per class (adjust freely)
)

 GradCAM++ summary grid saved → kaggle/working/dermavision_outputs/gradcam/gradcam_summary_grid.png
 Individual GradCAM++ images saved in → kaggle/working/dermavision_outputs/gradcam


##  Cell 12  Final Summary

In [13]:
print('\n' + '='*60)
print('    DermaVision AI  All Outputs Saved')
print('='*60)

saved_files = list(OUTPUT_DIR.rglob('*'))
for f in sorted(saved_files):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f'   {str(f.relative_to(OUTPUT_DIR)):55s}  ({size_kb:7.1f} KB)')

print('='*60)
print(f'\n   Final Test Metrics:')
for k, v in metrics_dict.items():
    print(f'      {k:<28}: {v}')
print('='*60)

# MLFLOW Cerrar la ejecución
mlflow.end_run()
print(f" Run {RUN_ID} cerrado correctamente.")


    DermaVision AI  All Outputs Saved
   confusion_matrix/confusion_matrix.png                    (  189.7 KB)
   curves/accuracy_loss_curve.png                           (   98.6 KB)
   gradcam/gradcam_summary_grid.png                         (  145.5 KB)
   metrics/classification_report.csv                        (    0.5 KB)
   metrics/classification_report_nuevo.csv                  (    0.8 KB)
   metrics/test_metrics.json                                (    0.2 KB)
   metrics/training_history.csv                             (    0.6 KB)
   model/best_coatnet.h5                                    (642493.0 KB)
   model/best_mobilenet.h5                                  (28370.9 KB)
   model/dermavision_1505_2154.h5                           (28370.9 KB)
   model/dermavision_1505_2220.h5                           (28370.9 KB)
   model/dermavision_1505_2256.h5                           (28370.9 KB)
   model/dermavision_1605_0041.h5                           (100967.9 KB)
   model/d